# ISIC 2024 - Training, tuning and deploying on SageMaker

The models were developed locally in notebooks 03-06. This notebook does the AWS part with the same code as
scripts (`src/train_cnn.py`, `src/train_lgbm.py`, `src/inference.py`):

1. Upload the data to S3
2. CNN training job on a GPU spot instance (5 folds)
3. LightGBM training job with the default parameters
4. Hyperparameter tuning for LightGBM with the pAUC as the objective
5. Blend the two models with the out-of-fold predictions from SageMaker
6. Deploy the blended model to an endpoint and test it
7. Delete the endpoint and check the cost
8. Submit the final model to Kaggle

In [1]:
import base64
import json
import os
import sys
import tarfile
import warnings

os.environ['SAGEMAKER_SUPPRESS_V2_WARNING'] = '1'
warnings.filterwarnings('ignore')

import boto3
import h5py
import numpy as np
import pandas as pd
import sagemaker
from sagemaker.deserializers import JSONDeserializer
from sagemaker.pytorch import PyTorch, PyTorchModel
from sagemaker.serializers import JSONSerializer
from sagemaker.tuner import ContinuousParameter, HyperparameterTuner, IntegerParameter

sys.path.insert(0, '../src')
from isic_pauc import p_auc_tpr

sess = sagemaker.Session()
sm = boto3.client('sagemaker')
s3 = boto3.client('s3')

bucket = sess.default_bucket()
prefix = 'isic-capstone'
role = boto3.client('iam').get_role(RoleName='SageMakerExecutionRole-isic')['Role']['Arn']

DATA_DIR = '../data'
IMG_PATH = r'C:\Users\xmuha\isic\data\train-image.hdf5'
N_FOLDS = 5

print(bucket)
print(sess.boto_region_name)

sagemaker-us-east-1-<account-id>
us-east-1


## 1. Upload the data to S3
The metadata (for LightGBM) and the images (for the CNN) go to separate folders, so every training job only
downloads what it needs. `folds.csv` is in both because both jobs use the same patient folds.

In [2]:
uploads = [
    (f'{DATA_DIR}/train-metadata.csv', 'metadata'),
    (f'{DATA_DIR}/test-metadata.csv', 'metadata'),
    (f'{DATA_DIR}/folds.csv', 'metadata'),
    (f'{DATA_DIR}/folds.csv', 'images'),
    (IMG_PATH, 'images'),
]
for path, folder in uploads:
    sess.upload_data(path, bucket=bucket, key_prefix=f'{prefix}/{folder}')

s3_metadata = f's3://{bucket}/{prefix}/metadata'
s3_images = f's3://{bucket}/{prefix}/images'

for obj in s3.list_objects_v2(Bucket=bucket, Prefix=prefix + '/')['Contents']:
    if '/output/' not in obj['Key']:
        print(f"{obj['Size'] / 1e6:8.1f} MB  {obj['Key']}")

    11.6 MB  isic-capstone/images/folds.csv
  1297.3 MB  isic-capstone/images/train-image.hdf5
    11.6 MB  isic-capstone/metadata/folds.csv
     0.0 MB  isic-capstone/metadata/test-metadata.csv
   257.5 MB  isic-capstone/metadata/train-metadata.csv


## 2. CNN training job
`train_cnn.py` is the code from `04_cnn.ipynb` as a script: for every fold it trains a ResNet18 on all malignant
lesions + 20 benign per malignant, predicts the full validation fold and saves the fold model.

- `ml.g4dn.xlarge` is the cheapest GPU instance (T4), which is enough for ResNet18 on 128x128 images.
- Spot instances to save budget. The job is short, so it's fine if it gets interrupted and has to start again.
- The job is started with `wait=False`, so the LightGBM jobs can run while the CNN is training.

In [3]:
metric_definitions = [
    {'Name': 'cv_pauc', 'Regex': 'cv_pauc: ([0-9\\.]+);'},
    {'Name': 'cv_pauc_std', 'Regex': 'cv_pauc_std: ([0-9\\.]+);'},
    {'Name': 'cv_auc', 'Regex': 'cv_auc: ([0-9\\.]+);'},
]

cnn_estimator = PyTorch(
    entry_point='train_cnn.py',
    source_dir='../src',
    role=role,
    framework_version='2.5.1',
    py_version='py311',
    instance_count=1,
    instance_type='ml.g4dn.xlarge',
    use_spot_instances=True,
    max_run=2 * 3600,
    max_wait=3 * 3600,
    hyperparameters={'epochs': 5, 'lr': 1e-4, 'batch-size': 64, 'neg-per-pos': 20, 'img-size': 128},
    metric_definitions=metric_definitions,
    output_path=f's3://{bucket}/{prefix}/output',
    base_job_name='isic-cnn',
)

cnn_estimator.fit({'train': s3_images}, wait=False)
cnn_job = cnn_estimator.latest_training_job.name
print(cnn_job)

isic-cnn-2026-10-01-17-10-39-628


### No spot capacity
The spot job never started: for 25 minutes AWS only answered "Insufficient capacity error from EC2 while launching
instances, retrying!". GPU spot instances were not available, so I stopped the spot job and started the same job
on a normal on-demand `ml.g4dn.xlarge`. The LightGBM jobs below had no problem getting CPU spot instances.

In [4]:
d = sm.describe_training_job(TrainingJobName=cnn_job)
waited = (pd.Timestamp.now(tz='UTC') - pd.Timestamp(d['CreationTime'])).total_seconds() / 60
print(f"{cnn_job}: {d['SecondaryStatus']} for {waited:.0f} minutes")
print(d['SecondaryStatusTransitions'][-1]['StatusMessage'])

sm.stop_training_job(TrainingJobName=cnn_job)

cnn_estimator = PyTorch(
    entry_point='train_cnn.py',
    source_dir='../src',
    role=role,
    framework_version='2.5.1',
    py_version='py311',
    instance_count=1,
    instance_type='ml.g4dn.xlarge',
    max_run=2 * 3600,
    hyperparameters={'epochs': 5, 'lr': 1e-4, 'batch-size': 64, 'neg-per-pos': 20, 'img-size': 128},
    metric_definitions=metric_definitions,
    output_path=f's3://{bucket}/{prefix}/output',
    base_job_name='isic-cnn',
)

cnn_estimator.fit({'train': s3_images}, wait=False)
cnn_job = cnn_estimator.latest_training_job.name
print(cnn_job)

isic-cnn-2026-10-01-17-10-39-628: Starting for 25 minutes
Insufficient capacity error from EC2 while launching instances, retrying!
isic-cnn-2026-10-01-17-35-29-090


## 3. LightGBM training job (default parameters)
`train_lgbm.py` is the cross-validation from `03_lightgbm.ipynb` as a script, with the v3 features from
`src/features.py`. LightGBM only needs a CPU, so it runs on an `ml.m5.xlarge` spot instance. It should give
the same CV pAUC as locally (0.1644).

In [5]:
def job_summary(job_name):
    d = sm.describe_training_job(TrainingJobName=job_name)
    metrics = {m['MetricName']: round(m['Value'], 4) for m in d.get('FinalMetricDataList', [])}
    return {
        'job': job_name,
        'status': d['TrainingJobStatus'],
        'instance': d['ResourceConfig']['InstanceType'],
        'train_seconds': d.get('TrainingTimeInSeconds'),
        'billable_seconds': d.get('BillableTimeInSeconds'),
        **metrics,
    }


lgbm_estimator = PyTorch(
    entry_point='train_lgbm.py',
    source_dir='../src',
    role=role,
    framework_version='2.5.1',
    py_version='py311',
    instance_count=1,
    instance_type='ml.m5.xlarge',
    use_spot_instances=True,
    max_run=3600,
    max_wait=2 * 3600,
    hyperparameters={'learning-rate': 0.05, 'n-estimators': 300, 'num-leaves': 31, 'min-child-samples': 20,
                     'colsample-bytree': 0.8, 'reg-lambda': 0.0},
    metric_definitions=metric_definitions,
    output_path=f's3://{bucket}/{prefix}/output',
    base_job_name='isic-lgbm',
)

lgbm_estimator.fit({'train': s3_metadata}, logs='None')
lgbm_job = lgbm_estimator.latest_training_job.name
job_summary(lgbm_job)

To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


2026-10-01 17:10:50 Starting - Starting the training job..2026-10-01 17:11:06 Starting - Preparing the instances for training.....
2026-10-01 17:11:38 Downloading - Downloading input data......
2026-10-01 17:12:13 Downloading - Downloading the training image.......
2026-10-01 17:12:59 Training - Training image download completed. Training in progress........
2026-10-01 17:13:39 Uploading - Uploading generated training model..2026-10-01 17:13:52 Completed - Training job completed

{'job': 'isic-lgbm-2026-10-01-17-10-43-246',
 'status': 'Completed',
 'instance': 'ml.m5.xlarge',
 'train_seconds': 134,
 'billable_seconds': 36,
 'cv_pauc': 0.1644,
 'cv_pauc_std': 0.0079,
 'cv_auc': 0.9545}

## 4. Hyperparameter tuning for LightGBM
Bayesian search over 6 parameters with the CV pAUC as the objective, 12 jobs with 3 running at the same time.

Every job runs the full 5-fold cross-validation, so the tuner compares mean pAUCs and not single folds. Still,
picking the best of 12 results on the same folds is optimistic, so I only switch to the tuned parameters if they
beat the default by more than 0.002. Smaller gains are far inside the fold-to-fold std (about 0.008) and would
just mean picking a lucky configuration.

In [6]:
hyperparameter_ranges = {
    'learning-rate': ContinuousParameter(0.01, 0.1, scaling_type='Logarithmic'),
    'n-estimators': IntegerParameter(150, 600),
    'num-leaves': IntegerParameter(8, 64),
    'min-child-samples': IntegerParameter(10, 100),
    'colsample-bytree': ContinuousParameter(0.4, 1.0),
    'reg-lambda': ContinuousParameter(0.0, 10.0),
}

tuner = HyperparameterTuner(
    estimator=lgbm_estimator,
    objective_metric_name='cv_pauc',
    objective_type='Maximize',
    hyperparameter_ranges=hyperparameter_ranges,
    metric_definitions=metric_definitions,
    max_jobs=12,
    max_parallel_jobs=3,
    base_tuning_job_name='isic-lgbm-hpo',
)

tuner.fit({'train': s3_metadata}, wait=False)
tuner_name = tuner.latest_tuning_job.name
print(tuner_name)
tuner.wait()

isic-lgbm-hpo-261001-2213
............................................................


In [7]:
hpo = tuner.analytics().dataframe()
hpo = hpo[hpo['TrainingJobStatus'] == 'Completed'].sort_values('FinalObjectiveValue', ascending=False)
cols = ['FinalObjectiveValue', 'learning-rate', 'n-estimators', 'num-leaves', 'min-child-samples',
        'colsample-bytree', 'reg-lambda', 'TrainingElapsedTimeSeconds']
hpo[cols].round(4).reset_index(drop=True)

,FinalObjectiveValue,learning-rate,n-estimators,num-leaves,min-child-samples,colsample-bytree,reg-lambda,TrainingElapsedTimeSeconds
0,0.1624,0.0211,479.0,14.0,33.0,0.6960,2.0743,139.0
1,0.1623,0.0132,437.0,15.0,38.0,0.6964,0.0404,139.0
2,0.1617,0.0152,518.0,50.0,93.0,0.6457,0.4355,134.0
3,0.1617,0.0251,460.0,28.0,38.0,0.7560,4.5873,132.0
4,0.1615,0.0264,454.0,9.0,34.0,0.4165,2.1989,129.0
5,0.1615,0.0215,356.0,27.0,85.0,0.6248,2.6433,134.0
6,0.1614,0.0629,402.0,22.0,44.0,0.7227,1.3516,139.0
7,0.1612,0.0232,388.0,12.0,52.0,0.4699,9.8786,140.0
8,0.1607,0.0178,468.0,49.0,90.0,0.5721,7.1727,129.0
9,0.1604,0.0395,465.0,8.0,89.0,0.9690,1.8924,136.0


In [8]:
MIN_GAIN = 0.002

default_pauc = job_summary(lgbm_job)['cv_pauc']
best_hpo_job = tuner.best_training_job()
best_hpo_pauc = job_summary(best_hpo_job)['cv_pauc']

use_tuned = best_hpo_pauc - default_pauc > MIN_GAIN
final_lgbm_job = best_hpo_job if use_tuned else lgbm_job

print(f'default: {default_pauc:.4f}')
print(f'best HPO: {best_hpo_pauc:.4f}  (gain {best_hpo_pauc - default_pauc:+.4f})')
print('using the', 'tuned' if use_tuned else 'default', 'parameters ->', final_lgbm_job)

default: 0.1644
best HPO: 0.1624  (gain -0.0020)
using the default parameters -> isic-lgbm-2026-10-01-17-10-43-246


The LightGBM job on SageMaker gives exactly the same CV pAUC as locally (0.1644). None of the 12 tuned
configurations beat the default parameters: the best one reached 0.1624, and all of them are between 0.158 and
0.162. So the default parameters stay, and the tuning mostly shows that the model is not very sensitive to these
parameters. Thanks to spot instances every LightGBM job was only billed for about 35-45 of its ~135 seconds.

## 5. CNN job results
Waiting for the on-demand CNN job from step 2.

In [9]:
sess.wait_for_job(cnn_job)
job_summary(cnn_job)

2026-10-01 17:42:29 Starting - Starting the training job
2026-10-01 17:42:29 Pending - Preparing the instances for training
2026-10-01 17:42:29 Downloading - Downloading the training image
2026-10-01 17:42:29 Training - Training image download completed. Training in progress................................................................
2026-10-01 17:50:24 Uploading - Uploading generated training model....
2026-10-01 17:50:47 Completed - Training job completed

{'job': 'isic-cnn-2026-10-01-17-35-29-090',
 'status': 'Completed',
 'instance': 'ml.g4dn.xlarge',
 'train_seconds': 736,
 'billable_seconds': 736,
 'cv_pauc': 0.1315,
 'cv_pauc_std': 0.0213,
 'cv_auc': 0.9076}

## 6. Blend with the out-of-fold predictions from SageMaker
Both jobs save their out-of-fold predictions to S3 (`output.tar.gz`). Blending them the same way as in
`05_blend.ipynb` (ranks within each fold, 70% LightGBM + 30% CNN) shows if the cloud run reproduces the local
result (0.1691).

In [10]:
def download_and_extract(job_name, filename, target):
    os.makedirs(target, exist_ok=True)
    key = f'{prefix}/output/{job_name}/output/{filename}'
    local = os.path.join(target, filename)
    s3.download_file(bucket, key, local)
    with tarfile.open(local) as tar:
        tar.extractall(target)
    os.remove(local)


WORK_DIR = r'C:\Users\xmuha\isic\sagemaker_run'
download_and_extract(final_lgbm_job, 'output.tar.gz', f'{WORK_DIR}/oof')
download_and_extract(cnn_job, 'output.tar.gz', f'{WORK_DIR}/oof')

oof = pd.read_csv(f'{WORK_DIR}/oof/oof_lgbm.csv').rename(columns={'oof': 'lgbm'})
oof = oof.merge(pd.read_csv(f'{WORK_DIR}/oof/oof_cnn.csv')[['isic_id', 'oof_cnn']].rename(columns={'oof_cnn': 'cnn'}),
                on='isic_id')

W = 0.7
oof['lgbm_rank'] = oof.groupby('fold')['lgbm'].rank(pct=True)
oof['cnn_rank'] = oof.groupby('fold')['cnn'].rank(pct=True)
oof['blend'] = W * oof['lgbm_rank'] + (1 - W) * oof['cnn_rank']

results = pd.DataFrame({
    col: [p_auc_tpr(oof.loc[oof['fold'] == f, 'target'], oof.loc[oof['fold'] == f, col], 0.80) for f in range(N_FOLDS)]
    for col in ['lgbm', 'cnn', 'blend']
}, index=[f'fold {i}' for i in range(N_FOLDS)])
results.loc['mean'] = results.iloc[:N_FOLDS].mean()
results.loc['std'] = results.iloc[:N_FOLDS].std(ddof=0)
results.round(4)

,lgbm,cnn,blend
fold 0,0.1688,0.1523,0.1740
fold 1,0.1529,0.1433,0.1592
fold 2,0.1689,0.1464,0.1732
fold 3,0.1741,0.1205,0.1710
fold 4,0.1573,0.0948,0.1517
mean,0.1644,0.1315,0.1658
std,0.0079,0.0213,0.0088


- LightGBM is identical to the local run (0.1644).
- The CNN job on SageMaker scored 0.1315 +/- 0.0213, lower than my local run (0.1432 +/- 0.0134), mostly on folds
  3 and 4. The code and settings are the same, but the GPU, the PyTorch version and the order of the random
  augmentations are different. With only about 80 malignant lesions per validation fold and a short training,
  the CNN result changes quite a bit from run to run. It is the less stable of the two models.
- Because of the weaker CNN run, the blend is 0.1658 +/- 0.0088 here instead of 0.1691 locally. It is still above
  LightGBM alone (better on 3 of 5 folds), but the gain depends on how good the CNN run is. Training the CNN with
  several seeds and averaging them would make this more stable.

## 7. Package the model
The endpoint needs everything in one `model.tar.gz`:
- the 5 LightGBM fold models and the 5 CNN fold models (the predictions of the 5 models are averaged)
- `reference.npz`: the sorted out-of-fold scores of both models. The blend works on ranks, so for a new lesion
  the endpoint looks up where its score would be in these scores (a number between 0 and 1).
- `blend.json`: the blend weight

`inference.py` and `features.py` are added by SageMaker from `source_dir`.

In [11]:
def download_model(job_name, target):
    os.makedirs(target, exist_ok=True)
    key = f'{prefix}/output/{job_name}/output/model.tar.gz'
    local = os.path.join(target, 'tmp_model.tar.gz')
    s3.download_file(bucket, key, local)
    with tarfile.open(local) as tar:
        tar.extractall(target)
    os.remove(local)


model_dir = f'{WORK_DIR}/model'
download_model(final_lgbm_job, model_dir)
download_model(cnn_job, model_dir)

np.savez(f'{model_dir}/reference.npz', lgbm=np.sort(oof['lgbm'].to_numpy()), cnn=np.sort(oof['cnn'].to_numpy()))
with open(f'{model_dir}/blend.json', 'w') as f:
    json.dump({'w': W}, f)

with tarfile.open(f'{WORK_DIR}/model.tar.gz', 'w:gz') as tar:
    for name in sorted(os.listdir(model_dir)):
        tar.add(os.path.join(model_dir, name), arcname=name)

model_data = sess.upload_data(f'{WORK_DIR}/model.tar.gz', bucket=bucket, key_prefix=f'{prefix}/model')
print(sorted(os.listdir(model_dir)))
print(model_data)

['blend.json', 'cnn_fold0.pt', 'cnn_fold1.pt', 'cnn_fold2.pt', 'cnn_fold3.pt', 'cnn_fold4.pt', 'lgbm_fold0.txt', 'lgbm_fold1.txt', 'lgbm_fold2.txt', 'lgbm_fold3.txt', 'lgbm_fold4.txt', 'lgbm_meta.json', 'reference.npz']
s3://sagemaker-us-east-1-<account-id>/isic-capstone/model/model.tar.gz


## 8. Deploy the endpoint
A CPU instance (`ml.m5.large`) is enough, ResNet18 on 128x128 images is fast even without a GPU.

In [12]:
model = PyTorchModel(
    model_data=model_data,
    role=role,
    entry_point='inference.py',
    source_dir='../src',
    framework_version='2.5.1',
    py_version='py311',
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name='isic-blend',
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)
print(predictor.endpoint_name)

-------!
isic-blend


## 9. Test the endpoint
The endpoint gets all lesions of one patient (metadata + image as base64), because the "ugly duckling" features
compare every lesion to the patient's other lesions.

This is only a test that the endpoint works. These patients were part of the training data of 4 of the 5 fold
models, so the result here is not an evaluation of the model (that is the cross-validation above).

In [13]:
train = pd.read_csv(f'{DATA_DIR}/train-metadata.csv', low_memory=False)
test = pd.read_csv(f'{DATA_DIR}/test-metadata.csv')
meta = train[list(test.columns) + ['target']]   # only the columns that exist at prediction time (+ target to check)


def patient_payload(patient_id):
    rows = meta[meta['patient_id'] == patient_id]
    lesions = []
    with h5py.File(IMG_PATH, 'r') as h5:
        for _, row in rows.iterrows():
            lesion = {k: (None if pd.isna(v) else v) for k, v in row.drop('target').items()}
            lesion['image'] = base64.b64encode(bytes(h5[row['isic_id']][()])).decode()
            lesions.append(lesion)
    return {'lesions': lesions}, rows


counts = meta.groupby('patient_id')['target'].agg(['size', 'sum'])
test_patients = counts[(counts['sum'] >= 1) & (counts['size'].between(100, 200))].index[:2].tolist()
test_patients

['IP_0076153', 'IP_0152575']

In [14]:
for patient_id in test_patients:
    payload, rows = patient_payload(patient_id)
    response = predictor.predict(payload)

    res = pd.DataFrame(response).merge(rows[['isic_id', 'target']], on='isic_id')
    res['rank_in_patient'] = res['risk_score'].rank(ascending=False).astype(int)

    print(f"{patient_id}: {len(res)} lesions, {int(res['target'].sum())} malignant, "
          f"payload {len(json.dumps(payload)) / 1e6:.2f} MB")
    print('rank of the malignant lesion(s):', res.loc[res['target'] == 1, 'rank_in_patient'].tolist(), 'of', len(res))
    display(res.sort_values('risk_score', ascending=False).head(5).round(4))

IP_0076153: 114 lesions, 1 malignant, payload 0.55 MB
rank of the malignant lesion(s): [1] of 114


,isic_id,lgbm_score,cnn_score,lgbm_rank,cnn_rank,risk_score,target,rank_in_patient
56,ISIC_5879031,0.7973,0.1673,0.9914,0.9616,0.9825,1,1
107,ISIC_9785127,0.4047,0.2630,0.9830,0.9765,0.9810,0,2
40,ISIC_4238043,0.9270,0.0635,0.9948,0.9153,0.9709,0,3
66,ISIC_6818036,0.0276,0.2133,0.9487,0.9702,0.9552,0,4
112,ISIC_9989863,0.0486,0.1153,0.9578,0.9462,0.9543,0,5


IP_0152575: 130 lesions, 1 malignant, payload 0.70 MB
rank of the malignant lesion(s): [2] of 130


,isic_id,lgbm_score,cnn_score,lgbm_rank,cnn_rank,risk_score,target,rank_in_patient
68,ISIC_4489084,0.9806,0.3313,0.9974,0.9827,0.9930,0,1
123,ISIC_9266851,0.9635,0.2973,0.9964,0.9800,0.9915,1,2
55,ISIC_3673812,0.8959,0.2296,0.9938,0.9725,0.9874,0,3
128,ISIC_9887634,0.4866,0.2009,0.9851,0.9683,0.9801,0,4
107,ISIC_8295004,0.1823,0.2710,0.9747,0.9773,0.9755,0,5


The endpoint works: for the first patient the malignant lesion is ranked 1st of 114 lesions, for the second
patient 2nd of 130. Each request has all lesions of a patient with their images (0.5-0.7 MB) and comes back as
one risk score per lesion, so a doctor could look at the lesions with the highest scores first.

## 10. Delete the endpoint
The endpoint costs money for every hour it runs, so it gets deleted right after the test.

In [15]:
# the model has to be deleted first, delete_model() looks it up through the endpoint config
predictor.delete_model()
predictor.delete_endpoint()

print('endpoints:', [e['EndpointName'] for e in sm.list_endpoints()['Endpoints']])
print('models:', [m['ModelName'] for m in sm.list_models()['Models']])
print('training jobs still running:',
      [j['TrainingJobName'] for j in sm.list_training_jobs(StatusEquals='InProgress')['TrainingJobSummaries']])

endpoints: ['isic-blend']
models: []
training jobs still running: []


In [16]:
# deleting takes a moment, the endpoint was still in "Deleting" state above. Checking again:
print('endpoints:', [(e['EndpointName'], e['EndpointStatus']) for e in sm.list_endpoints()['Endpoints']])
print('endpoint configs:', [c['EndpointConfigName'] for c in sm.list_endpoint_configs()['EndpointConfigs']])
print('models:', [m['ModelName'] for m in sm.list_models()['Models']])

endpoints: []
endpoint configs: []
models: []


## 11. Cost
Billable seconds of all training jobs (spot) and the on-demand price per hour in us-east-1.

In [17]:
PRICE_PER_HOUR = {'ml.g4dn.xlarge': 0.736, 'ml.m5.xlarge': 0.23}

hpo_jobs = tuner.analytics().dataframe()['TrainingJobName'].tolist()
jobs = pd.DataFrame([job_summary(j) for j in [cnn_job, lgbm_job] + hpo_jobs])
jobs['on_demand_cost'] = jobs['train_seconds'] / 3600 * jobs['instance'].map(PRICE_PER_HOUR)
jobs['spot_cost'] = jobs['billable_seconds'] / 3600 * jobs['instance'].map(PRICE_PER_HOUR)

print(f"training jobs: {len(jobs)}")
print(f"on-demand cost: ${jobs['on_demand_cost'].sum():.2f}")
print(f"cost with spot: ${jobs['spot_cost'].sum():.2f}  (saved {100 * (1 - jobs['spot_cost'].sum() / jobs['on_demand_cost'].sum()):.0f}%)")
jobs[['job', 'instance', 'train_seconds', 'billable_seconds', 'cv_pauc', 'spot_cost']].round(4)

training jobs: 14
on-demand cost: $0.26
cost with spot: $0.18  (saved 31%)


,job,instance,train_seconds,billable_seconds,cv_pauc,spot_cost
0,isic-cnn-2026-10-01-17-35-29-090,ml.g4dn.xlarge,736,736,0.1315,0.1505
1,isic-lgbm-2026-10-01-17-10-43-246,ml.m5.xlarge,134,36,0.1644,0.0023
2,isic-lgbm-hpo-261001-2213-012-e6fd3cd6,ml.m5.xlarge,136,44,0.1604,0.0028
3,isic-lgbm-hpo-261001-2213-011-564af20b,ml.m5.xlarge,129,35,0.1615,0.0022
4,isic-lgbm-hpo-261001-2213-010-5010ef21,ml.m5.xlarge,134,35,0.1617,0.0022
5,isic-lgbm-hpo-261001-2213-009-b2081b03,ml.m5.xlarge,129,34,0.1607,0.0022
6,isic-lgbm-hpo-261001-2213-008-03d7dbf3,ml.m5.xlarge,139,45,0.1614,0.0029
7,isic-lgbm-hpo-261001-2213-007-53eb6eba,ml.m5.xlarge,139,45,0.1624,0.0029
8,isic-lgbm-hpo-261001-2213-006-69eb5f2e,ml.m5.xlarge,129,34,0.1583,0.0022
9,isic-lgbm-hpo-261001-2213-005-fd85b434,ml.m5.xlarge,132,43,0.1617,0.0027


All 14 training jobs together cost about $0.18. The CNN job is most of it ($0.15) because it had to run
on-demand. The 13 LightGBM jobs on spot instances only cost about 3 cents in total, and spot saved about 73% on
them. On top of that comes the endpoint (`ml.m5.large`, $0.115 per hour, running for a few minutes) and a few
cents for S3, so the whole AWS part stayed far below $1.

## 12. Kaggle submission
As a last check, the final model is scored on the hidden test set of the Kaggle competition (about 500,000 lesions
from new patients). ISIC 2024 is a code competition: you can't upload predictions, Kaggle runs a notebook without
internet on the hidden test set. So I did three things with the Kaggle CLI:

1. Uploaded the model files from step 7 (5 LightGBM + 5 CNN fold models, `lgbm_meta.json`, `blend.json`) as a
   private Kaggle dataset.
2. Pushed `src/kaggle_inference.py` as a Kaggle notebook (GPU on, internet off). It computes the same features,
   averages the 5 fold models of each type and blends the ranks with 70% LightGBM + 30% CNN.
3. Submitted that notebook to the competition.

```
kaggle datasets create -p model
kaggle kernels push -p kernel
kaggle competitions submit isic-2024-challenge -k rizwanjx/isic-2024-blend-inference -v 2 -f submission.csv -m "..."
```

The first version of the notebook failed because the competition data is mounted in a different folder than I
expected (`/kaggle/input/competitions/isic-2024-challenge`), so the script now looks the folders up.

In [18]:
import subprocess

kaggle = os.path.join(os.path.dirname(sys.executable), 'kaggle')
out = subprocess.run([kaggle, 'competitions', 'submissions', 'isic-2024-challenge', '--csv'],
                     capture_output=True, text=True).stdout
from io import StringIO
subs = pd.read_csv(StringIO(out))
subs[['date', 'status', 'publicScore', 'privateScore']]

,date,status,publicScore,privateScore
0,2026-10-01 19:55:12.433000,SubmissionStatus.COMPLETE,0.17058,0.15902
1,2026-10-01 19:44:17.480000,SubmissionStatus.COMPLETE,0.17058,0.15902


The final model scores **0.1706 on the public part and 0.1590 on the private part** of the hidden test set (the
two submissions are the same notebook). My cross-validation (0.166 to 0.169) is between the two. A silver medal on the private part starts at about 0.1705. So the model works on new patients about as well as the cross-validation said, but it is not close
to the best solutions.